In [2]:
import tensorflow as tf

In [3]:
print(tf.__version__)

2.19.0


In [4]:
import cv2
print("OpenCV version:", cv2.__version__)

OpenCV version: 4.11.0


In [5]:
import numpy as np
import os
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import img_to_array
from sklearn.model_selection import train_test_split

In [6]:
IMG_SIZE = 48
DATASET_PATH = r"C:\Users\monis\Downloads\training"

def load_data():
    data = []
    labels = []
    
    for label in os.listdir(DATASET_PATH):
        folder_path = os.path.join(DATASET_PATH, label)
        
        if not os.path.isdir(folder_path):
            continue  # skip if it's not a directory

        for file in os.listdir(folder_path):
            img_path = os.path.join(folder_path, file)
            try:
                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                if img is None:
                    continue
                img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
                data.append(img)
                labels.append(int(label))
            except Exception as e:
                print(f"Failed to load {img_path}: {e}")
    
    return np.array(data), np.array(labels)

In [7]:
data, labels = load_data()
print("Total images loaded:", len(data))
print("Total labels loaded:", len(labels))

Total images loaded: 7178
Total labels loaded: 7178


In [8]:
from sklearn.model_selection import train_test_split
import numpy as np

# Convert to NumPy arrays (if not already)
data = np.array(data) / 255.0  # normalize pixel values to [0, 1]
labels = np.array(labels)

# Reshape images to add channel dimension (grayscale)
data = data.reshape(-1, 48, 48, 1)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    data, labels, test_size=0.2, random_state=42)

print(f"Train samples: {len(X_train)}, Test samples: {len(X_test)}")

Train samples: 5742, Test samples: 1436


In [9]:
def build_model():
    model = models.Sequential([
        layers.Conv2D(32, (3,3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 1)),
        layers.MaxPooling2D(2,2),
        layers.Conv2D(64, (3,3), activation='relu'),
        layers.MaxPooling2D(2,2),
        layers.Conv2D(128, (3,3), activation='relu'),
        layers.MaxPooling2D(2,2),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(CLASSES, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

In [10]:
from tensorflow.keras import models, layers
MODEL_PATH = "emotion_model.h5"
CLASSES = 7
if os.path.exists(MODEL_PATH):
    model = tf.keras.models.load_model(MODEL_PATH)
    print("✅ Model loaded from disk.")
else:
    model = build_model()
    model.fit(X_train, y_train, epochs=10, validation_data=(X_test, y_test))
    model.save(MODEL_PATH)
    print("✅ Model trained and saved.")

c:\Users\monis\anaconda3\envs\tf_env\lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
180/180 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - accuracy: 0.2072 - loss: 1.8665 - val_accuracy: 0.2611 - val_loss: 1.7954
Epoch 2/10
180/180 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.2552 - loss: 1.7998 - val_accuracy: 0.3322 - val_loss: 1.7086
Epoch 3/10
180/180 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.3296 - loss: 1.6961 - val_accuracy: 0.3280 - val_loss: 1.6484
Epoch 4/10
180/180 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.3686 - loss: 1.6034 - val_accuracy: 0.3830 - val_loss: 1.5933
Epoch 5/10
180/180 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.4045 - loss: 1.5569 - val_accuracy: 0.4185 - val_loss: 1.5542
Epoch 6/10
180/180 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.4110 - loss: 1.4915 - val_accuracy: 0.3747 - val_loss: 1.5914
Epoch 7/10
180/180 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.4608 - loss: 1.3980 - val_accuracy: 0.4248 - val_loss: 1.4795
Epoch 8/10
180/180 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.4627 - loss: 1.3867 - val_accu

✅ Model trained and saved.


In [11]:
import cv2
import numpy as np
import tensorflow as tf

# Load your trained model
model = tf.keras.models.load_model("emotion_model.h5")

# Define emotion labels (based on your class folder renaming)
emotion_labels = {
    0: "Angry", 1: "Disgust", 2: "Fear",
    3: "Happy", 4: "Sad", 5: "Surprise", 6: "Neutral"
}

# Initialize webcam
cap = cv2.VideoCapture(0)
IMG_SIZE = 48

# Load Haar Cascade for face detection
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Convert to grayscale (your model was trained on gray images)
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Detect faces in the frame
    faces = face_cascade.detectMultiScale(gray, 1.3, 5)

    for (x, y, w, h) in faces:
        roi_gray = gray[y:y+h, x:x+w]
        roi_resized = cv2.resize(roi_gray, (IMG_SIZE, IMG_SIZE)) / 255.0
        roi_reshaped = np.reshape(roi_resized, (1, IMG_SIZE, IMG_SIZE, 1))

        # Predict emotion
        prediction = model.predict(roi_reshaped)
        emotion = emotion_labels[np.argmax(prediction)]

        # Draw bounding box and label
        cv2.rectangle(frame, (x, y), (x+w, y+h), (255, 0, 0), 2)
        cv2.putText(frame, emotion, (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (36,255,12), 2)

    # Show frame
    cv2.imshow("Emotion Detection", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 140ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━